# Export newest_results to CSV

This notebook reads every `results_*_epochs_*_seed_*_lr_*_polydegree_*.npz` file under `newest_results`, summarizes each burn-in result, sorts rows by `true_r2` then seed, and writes one CSV file.

In [ ]:
import csv
import re
from datetime import datetime
from pathlib import Path

import numpy as np

THIS_DIR = Path.cwd()
INPUT_DIR = THIS_DIR / "newest_results"
OUTPUT_DIR = THIS_DIR / "view_results_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_CSV = OUTPUT_DIR / f"newest_results_summary_{RUN_STAMP}.csv"

FILE_RE = re.compile(
    r"results_(?P<test_name>.+)_epochs_(?P<epochs>\d+)_seed_(?P<seed>\d+)_"
    r"lr_(?P<lr_label>[^_]+)_polydegree_(?P<poly_degree>\d+)\.npz$"
)

FIELDNAMES = [
    "true_r2",
    "test_name",
    "epochs",
    "seed",
    "lr",
    "poly_degree",
    "burnin_ratio",
    "saved_r2_train_mean",
    "saved_r2_test_mean",
    "mean_param_r2_train",
    "mean_param_r2_test",
    "M1 Accuracy",
    "M1 TPR",
    "M1 FPR",
    "M1 FDR",
    "M2 Accuracy",
    "M2 TPR",
    "M2 FPR",
    "M2 FDR",
    "selected_modality_1",
    "selected_modality_2",
    "source_file",
]

def parse_lr(label):
    return float(label.replace("p", ".").replace("m", "-"))

def scalar_from_npz(bundle, key, default=np.nan):
    if key not in bundle.files:
        return default
    value = bundle[key]
    if np.asarray(value).shape == ():
        return np.asarray(value).item()
    return value

def safe_divide(numerator, denominator):
    return float(numerator / denominator) if denominator > 0 else np.nan

def empty_selection_metrics():
    return {
        "Accuracy": np.nan,
        "TPR": np.nan,
        "FPR": np.nan,
        "FDR": np.nan,
    }

def selection_metrics_from_confusion_matrix(confusion_matrix_values):
    if confusion_matrix_values is None:
        return empty_selection_metrics()

    cm = np.asarray(confusion_matrix_values)
    if cm.shape != (2, 2):
        return empty_selection_metrics()

    tn, fp, fn, tp = [int(x) for x in cm.ravel()]
    total = tn + fp + fn + tp
    return {
        "Accuracy": safe_divide(tp + tn, total),
        "TPR": safe_divide(tp, tp + fn),
        "FPR": safe_divide(fp, fp + tn),
        "FDR": safe_divide(fp, fp + tp),
    }

def add_modality_metrics(row, result, modality_idx):
    metric_prefix = f"M{modality_idx + 1}"
    confusion_matrices = result.get("confusion_matrix_list", [])
    if modality_idx < len(confusion_matrices):
        metrics = selection_metrics_from_confusion_matrix(confusion_matrices[modality_idx])
    else:
        metrics = empty_selection_metrics()

    for metric_name, value in metrics.items():
        row[f"{metric_prefix} {metric_name}"] = value

def list_result_files(input_dir):
    result_files = []
    for path in sorted(input_dir.glob("results_*_epochs_*_seed_*_lr_*_polydegree_*.npz")):
        match = FILE_RE.match(path.name)
        if match is None:
            print(f"Skipping unrecognized result file name: {path.name}")
            continue
        result_files.append(
            {
                "test_name": match.group("test_name"),
                "epochs": int(match.group("epochs")),
                "seed": int(match.group("seed")),
                "lr": parse_lr(match.group("lr_label")),
                "poly_degree": int(match.group("poly_degree")),
                "path": path,
            }
        )
    return result_files

def summarize_file(file_info):
    rows = []
    with np.load(file_info["path"], allow_pickle=True) as bundle:
        true_r2 = float(scalar_from_npz(bundle, "true_r2"))
        test_name = str(scalar_from_npz(bundle, "test_name", file_info["test_name"]))
        results_by_burnin = bundle["results_by_burnin"].item()
        burnin_ratios = [float(x) for x in bundle["burnin_ratios"]]

        for burnin_ratio in burnin_ratios:
            result = results_by_burnin[str(float(burnin_ratio))]
            mask_list = result.get("mask_list", [])
            selected_counts = [int(np.asarray(mask).sum()) for mask in mask_list]
            mean_parameter_r2 = result["mean_parameter_r2"]

            row = {
                "true_r2": true_r2,
                "test_name": test_name,
                "epochs": file_info["epochs"],
                "seed": file_info["seed"],
                "lr": file_info["lr"],
                "poly_degree": file_info["poly_degree"],
                "burnin_ratio": burnin_ratio,
                "saved_r2_train_mean": float(np.nanmean(result["saved_r2_train"])),
                "saved_r2_test_mean": float(np.nanmean(result["saved_r2_test"])),
                "mean_param_r2_train": float(mean_parameter_r2["r2_train"]),
                "mean_param_r2_test": float(mean_parameter_r2["r2_test"]),
                "selected_modality_1": selected_counts[0] if len(selected_counts) > 0 else np.nan,
                "selected_modality_2": selected_counts[1] if len(selected_counts) > 1 else np.nan,
                "source_file": file_info["path"].name,
            }
            add_modality_metrics(row, result, modality_idx=0)
            add_modality_metrics(row, result, modality_idx=1)
            rows.append(row)
    return rows

print(f"Reading newest results from: {INPUT_DIR.resolve()}")
print(f"CSV will be written to: {OUTPUT_CSV.resolve()}")

In [ ]:
if not INPUT_DIR.exists():
    raise FileNotFoundError(f"Missing newest_results folder: {INPUT_DIR.resolve()}")

result_files = list_result_files(INPUT_DIR)
print(f"Found {len(result_files)} result files")

rows = []
for file_info in result_files:
    rows.extend(summarize_file(file_info))

# Group rows by true r2, with r2=0.3 first, then 0.5, then 0.8 for the current sweep.
# Within each r2 group, rows are ordered by burn-in ratio, seed 1-20, then run settings.
rows.sort(key=lambda row: (row["true_r2"], row["burnin_ratio"], row["seed"], row["lr"], row["poly_degree"]))

with OUTPUT_CSV.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=FIELDNAMES, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(rows)

print(f"Saved {len(rows)} rows to {OUTPUT_CSV.resolve()}")

# Quick sanity preview: the first 20 rows should be true_r2=0.3, seeds 1-20, when all seeds are present.
for row in rows[:20]:
    print(
        f"true_r2={row['true_r2']}, seed={row['seed']}, "
        f"epochs={row['epochs']}, lr={row['lr']}, poly_degree={row['poly_degree']}"
    )